# Brain Tumor — Kaggle GPU training

Runs the whole training pipeline on a Kaggle GPU. Everything before this point
(preprocessing, mask generation) is cheap on Kaggle's local disk and painfully
slow on a OneDrive-synced folder, so it all happens here.

**Before running, in the notebook sidebar:**

| Setting | Value |
|---|---|
| Accelerator | GPU T4 x2 (or P100) |
| Internet | On — needed for `git clone`, `pip install`, ImageNet weights |
| Persistence | Variables and files off (outputs are saved explicitly at the end) |

Budget: roughly 20 min detection + 40 min classification + 60 min segmentation,
well inside one 12 h session. Kaggle allows 30 GPU-hours per week.

You can run stages 1-2 and 3 in separate sessions; stage 3 only needs the masks,
not the TensorFlow models.

## 0. Environment check

Fail fast if the accelerator is off — otherwise this silently runs for hours on CPU.

In [1]:
import tensorflow as tf, torch, keras
print('TF', tf.__version__, '| Keras', keras.__version__, '| Torch', torch.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('TF GPUs:', gpus)
print('Torch CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
assert gpus, 'No GPU. Set Accelerator to GPU in the sidebar before running.'

TF 2.20.0 | Keras 3.13.2 | Torch 2.10.0+cu128
TF GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
Torch CUDA: True Tesla T4


## 1. Get the code and data

`Dataset/` is committed to the repo, so the clone brings the 7200 images with it
(~150 MB). Nothing to upload to Kaggle separately.

In [2]:
REPO_URL = 'https://github.com/Mounika-Reddy-0802/Brain-Tumor-Detection-and-Segmentation-.git'
REPO_DIR = 'Brain-Tumor-Detection-and-Segmentation-'

import os, pathlib
os.chdir('/kaggle/working')
if pathlib.Path(REPO_DIR).exists():
    # Already cloned earlier in this session: pull so a re-run picks up fixes
    # instead of silently reusing stale code.
    !cd $REPO_DIR && git fetch -q origin && git reset -q --hard origin/main
else:
    !git clone --depth 1 $REPO_URL
os.chdir(f'/kaggle/working/{REPO_DIR}')

print('cwd:', os.getcwd())
!git log --oneline -1
# src/models must be present; it was missing from early clones.
assert pathlib.Path('src/models/backbone.py').exists(), 'src/models missing - re-clone'
!ls Dataset/Training && du -sh Dataset

Cloning into 'Brain-Tumor-Detection-and-Segmentation-'...
remote: Enumerating objects: 7074, done.
remote: Counting objects: 100% (7074/7074), done.
remote: Compressing objects: 100% (7068/7068), done.
remote: Total 7074 (delta 4), reused 7054 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (7074/7074), 151.73 MiB | 42.42 MiB/s, done.
Resolving deltas: 100% (4/4), done.
Updating files: 100% (7244/7244), done.
cwd: /kaggle/working/Brain-Tumor-Detection-and-Segmentation-
5c0c2c6 (grafted, HEAD -> main, origin/main, origin/HEAD) Fix the U-Net encoder name and salvage partial evaluation runs
glioma	meningioma  notumor  pituitary
175M	Dataset


In [3]:
# smp is not preinstalled on Kaggle; everything else already is.
!pip install -q segmentation-models-pytorch
import segmentation_models_pytorch as smp; print('smp', smp.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.1 MB/s eta 0:00:0000:01
smp 0.5.0


## 2. Prepare data (~3 min)

Writes cropped + CLAHE-enhanced 224x224 images to `data/processed`, then the Otsu
pseudo-masks to `data/masks`. Doing this once means the training loops only decode
JPEGs instead of re-running OpenCV every epoch.

In [4]:
!python -m src.data.preprocessing --raw-dir Dataset --out-dir data/processed --img-size 224
!python -m scripts.generate_masks --config configs/config_processed.yaml
!find data/processed -name '*.jpg' | wc -l && find data/masks -name '*.png' | wc -l

Preprocessing complete: data/processed
Source images: data/processed
Mask output:   data/masks
Testing masks: 100%|███████████████████████| 1600/1600 [00:06<00:00, 234.29it/s]
Wrote 7200 masks, skipped 0 existing.
7200
7200


## 2b. Pre-flight check (~1 min)

Builds every model, pushes a tiny batch through each training path and verifies
the input conventions. Anything misconfigured fails here in under a minute rather
than twenty minutes into a training run — which is exactly how the `efficientnet_b3`
vs `efficientnet-b3` encoder-name bug cost a session.

**If this cell fails, fix the reported problem before continuing.**

In [5]:
!python -m scripts.smoke_test --config configs/config_processed.yaml


config: configs/config_processed.yaml
TF 2.20.0 | torch 2.10.0+cu128 | GPU: TF=2 torch=True

[data]
  ok    dataset resolves and splits are readable - 5600 training images under data/processed
I0000 00:00:1785975422.109870     168 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1785975422.113334     168 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
  ok    tf.data pipeline emits [0, 255] for EfficientNet - batch (4, 224, 224, 3), range [0.0, 255.0]
  ok    albumentations transforms accept every configured argument - no ignored arguments

[tensorflow]
43941136/43941136 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
2026-08-06 00:18:26.597358: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel ti

## 3. Detection — binary tumor / no-tumor (~20 min)

5 warmup epochs with the backbone frozen, then up to 32 fine-tuning epochs with
early stopping on `val_recall` (missed tumors matter more than false alarms).

In [6]:
!python -m src.training.train_tf_detection --config configs/config_processed.yaml

TF GPUs available: 2
PyTorch CUDA available: True
I0000 00:00:1785975718.217040     324 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1785975718.219471     324 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
Epoch 1/5
I0000 00:00:1785975792.394366     356 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
149/149 ━━━━━━━━━━━━━━━━━━━━ 258s 1s/step - accuracy: 0.8935 - auc: 0.9701 - loss: 0.2462 - recall: 0.8798 - val_accuracy: 0.9750 - val_auc: 0.9926 - val_loss: 0.0962 - val_recall: 0.9841
Epoch 2/5
149/149 ━━━━━━━━━━━━━━━━━━━━ 7s 47ms/step - accuracy: 0.9450 - auc: 0.9857 - loss: 0.1538 - recall: 0.9431 - val_accuracy: 0.9714 - val_auc: 0.9

## 4. Classification — 4-class (~40 min)

In [7]:
!python -m src.training.train_tf_classification --config configs/config_processed.yaml

TF GPUs available: 2
PyTorch CUDA available: True
I0000 00:00:1785976878.431001     988 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1785976878.433252     988 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
Epoch 1/5
I0000 00:00:1785976942.583337    1019 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
149/149 ━━━━━━━━━━━━━━━━━━━━ 229s 1s/step - accuracy: 0.7420 - loss: 0.7913 - top2_accuracy: 0.9038 - val_accuracy: 0.8524 - val_loss: 0.4258 - val_top2_accuracy: 0.9583
Epoch 2/5
149/149 ━━━━━━━━━━━━━━━━━━━━ 7s 44ms/step - accuracy: 0.8065 - loss: 0.5379 - top2_accuracy: 0.9458 - val_accuracy: 0.8464 - val_loss: 0.4148 - val_top2_accuracy:

## 5. Segmentation — U-Net (~60 min)

Trains against the precomputed weak labels: the brightest compact blob inside the
brain, with `notumor` scans given an empty mask. Roughly a third of those masks
land on the actual lesion; the rest catch eye globes, skull-base structures or
ventricles. Treat the resulting Dice as agreement with that heuristic, not as
tumour localisation accuracy.

For genuine segmentation quality this needs expert annotations — see the
segmentation note in the README.

In [8]:
!python -m src.training.train_torch_segmentation --config configs/config_processed.yaml

TF GPUs available: 2
PyTorch CUDA available: True
Using 5600 precomputed masks from data/masks
config.json: 100%|██████████████████████████████| 106/106 [00:00<00:00, 582kB/s]
model.safetensors: 100%|███████████████████| 49.3M/49.3M [00:01<00:00, 47.1MB/s]
Epoch 1/32: 100%|█████████████████████████████| 298/298 [01:10<00:00,  4.23it/s]
Epoch 01 | Train Loss: 0.7579 | Val Loss: 0.6726 | Val Dice: 0.0004
Epoch 2/32: 100%|█████████████████████████████| 298/298 [00:43<00:00,  6.78it/s]
Epoch 02 | Train Loss: 0.6442 | Val Loss: 0.6150 | Val Dice: 0.4506
Epoch 3/32: 100%|█████████████████████████████| 298/298 [00:43<00:00,  6.84it/s]
Epoch 03 | Train Loss: 0.5814 | Val Loss: 0.5383 | Val Dice: 0.5051
Epoch 4/32: 100%|█████████████████████████████| 298/298 [00:44<00:00,  6.75it/s]
Epoch 04 | Train Loss: 0.4973 | Val Loss: 0.4567 | Val Dice: 0.4843
Epoch 5/32: 100%|█████████████████████████████| 298/298 [00:43<00:00,  6.80it/s]
Epoch 05 | Train Loss: 0.4429 | Val Loss: 0.4206 | Val Dice: 0.510

## 6. Evaluate on the held-out Testing split

These are the numbers to quote in the report — the values printed during training are validation-split numbers.

In [9]:
!python -m src.evaluation.evaluate_all --config configs/config_processed.yaml
import json; print(json.dumps(json.load(open('outputs/evaluation_report.json')), indent=2))

TF GPUs available: 2
PyTorch CUDA available: True
I0000 00:00:1785979628.520413    2532 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1785979628.522665    2532 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 690 variables whereas the saved optimizer has 694 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
Detection evaluation: 100%|█████████████████████| 50/50 [01:09<00:00,  1.40s/it]
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 

## 7. Save the artifacts

`/kaggle/working` is wiped when the session ends. Commit the notebook (Save Version →
Save & Run All) so the output files persist, then download `trained_models.zip`.

In [10]:
import shutil, os
os.makedirs('/kaggle/working/artifacts', exist_ok=True)
for src in ['models', 'checkpoints', 'outputs', 'logs']:
    if os.path.exists(src):
        shutil.copytree(src, f'/kaggle/working/artifacts/{src}', dirs_exist_ok=True)
shutil.make_archive('/kaggle/working/trained_models', 'zip', '/kaggle/working/artifacts')
print(os.path.getsize('/kaggle/working/trained_models.zip') / 1e6, 'MB')
!ls -la /kaggle/working/artifacts/models

930.621777 MB
total 549244
drwxr-xr-x 2 root root      4096 Aug  6 01:26 .
drwxr-xr-x 6 root root      4096 Aug  6 01:29 ..
-rw-r--r-- 1 root root  53222795 Aug  6 01:20 best_unet.pth
-rw-r--r-- 1 root root 231941776 Aug  6 01:02 classification_model.keras
-rw-r--r-- 1 root root 224024294 Aug  6 00:41 detection_model.keras
-rw-r--r-- 1 root root  53222795 Aug  6 01:26 unet_last.pth


In [12]:
import os, shutil
import tensorflow as tf
from IPython.display import FileLink, display

os.makedirs('/kaggle/working/slim', exist_ok=True)

# compile=False drops the optimizer, so re-saving writes weights only
for name in ['detection_model', 'classification_model']:
    m = tf.keras.models.load_model(f'models/{name}.keras', compile=False)
    m.save(f'/kaggle/working/slim/{name}.keras')

shutil.copy('models/best_unet.pth', '/kaggle/working/slim/best_unet.pth')
shutil.copytree('outputs', '/kaggle/working/slim/outputs', dirs_exist_ok=True)
shutil.copytree('logs', '/kaggle/working/slim/logs', dirs_exist_ok=True)

shutil.make_archive('/kaggle/working/models_slim', 'zip', '/kaggle/working/slim')
print(round(os.path.getsize('/kaggle/working/models_slim.zip') / 1e6, 1), 'MB')
display(FileLink('/kaggle/working/models_slim.zip'))

134.1 MB


/kaggle/working/models_slim.zip